In [83]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [84]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [85]:
# query_com = """
# SELECT DISTINCT Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2026-09-19'
#     AND v.Fecha <= '2026-09-19'
#     AND Ubicacion = 'Social Selling'
#     AND v.Venta_Neta > 0
# GROUP BY  Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_com, engine)

# df_ventas.head()

In [86]:
# CONSULTAS CUMPLEAÑOS

query_data = """
SELECT DISTINCT Pedido,
    v.Cliente, 
	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
	cc.Email, cc.Celular, 
	v.Fecha AS Fecha_Venta,
	Canal,
    SUM(Venta_Neta) AS Venta 
FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'SEPTIEMBRE_2026')
	AND Fecha BETWEEN '2026-09-30' AND '2026-09-30'
    AND Ubicacion IN ('Social Selling') 
    AND V.Cliente NOT IN ('C22222222222')
GROUP BY  Pedido,
    v.Cliente,
    cc.Nombres + ' ' +  cc.Apellidos,
    cc.Email,
    cc.Celular,
    v.Fecha,
    v.Canal
"""


# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_data, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,104123,C31714225,NATALIA HENAO GALLEGO,nata.henao@gmail.com,3172993934,2026-09-30,CHATCENTER WEB,30881.03
1,104124,C3176417822,MICHELLE MARIA GREGORY MEDINA,michelledeferrer@gmail.com,3176417822,2026-09-30,CHATCENTER WEB,129196.13
2,104126,C19189694,JOSÉ LEONARDO SANSÓN G,leonardosanzon@gmail.com,3006157292,2026-09-30,CHATCENTER WEB,161967.83
3,104127,C66711820,LUZ PIEDAD CARDONA B.,luzpica@yahoo.com,3104349975,2026-09-30,CHATCENTER WEB,298096.42
4,104130,C1032399832,LINA LONDOÑO,linalond11@gmail.com,3227925162,2026-09-30,CHATCENTER WEB,118482.30


In [87]:
# Leer archivo
df_social = pd.read_excel("Atribucion.xlsx")

df_social['telefono'] = df_social['telefono'].astype(str).str[2:]

In [88]:
# Normalizar df_ventas
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por celular
df_atribucion = pd.merge(df_ventas, df_social, left_on='Celular', right_on='telefono' , how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.2f}")
# Resultado final
df_atribucion.head()

Total de Ventas: 16
Total de venta atribuida: 3,157,255.10


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,telefono
0,104123,C31714225,NATALIA HENAO GALLEGO,nata.henao@gmail.com,3172993934,2026-09-30,CHATCENTER WEB,30881.03,3172993934
1,104124,C3176417822,MICHELLE MARIA GREGORY MEDINA,michelledeferrer@gmail.com,3176417822,2026-09-30,CHATCENTER WEB,129196.13,3176417822
2,104126,C19189694,JOSÉ LEONARDO SANSÓN G,leonardosanzon@gmail.com,3006157292,2026-09-30,CHATCENTER WEB,161967.83,3006157292
3,104127,C66711820,LUZ PIEDAD CARDONA B.,luzpica@yahoo.com,3104349975,2026-09-30,CHATCENTER WEB,298096.42,3104349975
4,104130,C1032399832,LINA LONDOÑO,linalond11@gmail.com,3227925162,2026-09-30,CHATCENTER WEB,118482.30,3227925162


In [89]:
df_atribucion.to_excel('Ventas.xlsx', index=False)